In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
bronze_order_items_df = spark.table(
    "ecommerce_lakehouse.bronze.order_items_raw"
)

In [0]:
silver_order_items_df = bronze_order_items_df \
    .dropDuplicates([
        "order_id",
        "order_item_id"
    ]) \
    .withColumn(
        "shipping_limit_date",
        to_timestamp("shipping_limit_date")
    )

In [0]:
silver_order_items_df = silver_order_items_df.withColumn(
    "total_item_value",
    col("price") + col("freight_value")
)

In [0]:
silver_order_items_df = silver_order_items_df.withColumn(
    "shipping_percentage",
    round(
        (col("freight_value") / col("price")) * 100,
        2
    )
)

In [0]:
(
    silver_order_items_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            "ecommerce_lakehouse.silver.order_items_clean"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.silver.order_items_clean
LIMIT 10;